**Status: lap ingestion for 2023 onward from OpenF1.** Since mid-October 2026 F1's live-timing archive (`livetiming.formula1.com/static`) returns 401 without an F1 TV login, so `phase_1.ipynb`'s FastF1 lap loads fail for any race not already cached. This notebook appends every race since 2023 that is missing from `f1_master_lap_dataset`, built from the free OpenF1 API (`api.openf1.org`).

- Same table, same columns as `phase_1.ipynb`; fields OpenF1 does not have (speed at the finish line, qualifying lap times, deleted-lap flags) are null.
- Running position is the order of lap-completion times; on 2023 races this matches FastF1's position on 99.3% of laps, and stints, compounds and tyre age on about 99.5%.
- Safety car / VSC laps come from race-control messages (97.6% agreement with FastF1 on 2023). Both are stored as TrackStatus "4", which is what downstream `contains("[45]")` checks look for.
- Results, grid and qualifying position come from `f1_race_results` (`phase_1b_results.ipynb`), so run that first.
- Resumable: races already in the table are skipped. Re-run after every race weekend, then `phase_2` onward.
- `f1_race_control_messages` is not written here (nothing reads it yet).

In [ ]:
import json
import time
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

MASTER = "workspace.default.f1_master_lap_dataset"
RESULTS = "workspace.default.f1_race_results"
FIRST_YEAR = 2023
API = "https://api.openf1.org/v1/"

In [ ]:
def get(endpoint, **q):
    # throttled with retries: OpenF1 answers 429 on bursts
    url = API + endpoint + "?" + urllib.parse.urlencode(q)
    for attempt in range(6):
        try:
            time.sleep(0.7)
            with urllib.request.urlopen(url, timeout=60) as r:
                return pd.DataFrame(json.load(r))
        except Exception as e:
            print(f"   retry {endpoint}: {e}")
            time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"OpenF1 request failed: {url}")


def sc_laps(rc, n):
    # SC/VSC from deployment until an ending / in-this-lap message, the next deployment, or 6 laps
    sc = np.zeros(n + 2, bool)
    if rc.empty or "lap_number" not in rc:
        return sc
    on = None
    for r in rc.sort_values("date").itertuples():
        msg, lap = str(r.message).upper(), r.lap_number
        if pd.isna(lap):
            continue
        lap = int(lap)
        if "DEPLOYED" in msg and ("SAFETY CAR" in msg or "VSC" in msg):
            if on is not None:
                sc[on:min(lap, on + 6)] = True
            on = lap
        elif on is not None and ("ENDING" in msg or "IN THIS LAP" in msg):
            sc[on:min(lap, on + 6) + 1] = True
            on = None
    if on is not None:
        sc[on:on + 6] = True
    return sc


def cast(v, t):
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    return {"double": float, "bigint": int, "string": str, "boolean": bool,
            "timestamp": lambda x: pd.Timestamp(x).tz_localize(None).to_pydatetime() if pd.Timestamp(x).tzinfo else pd.Timestamp(x).to_pydatetime()}[t](v)

In [ ]:
schema = spark.table(MASTER).schema
master_keys = spark.table(MASTER).select("Year", "Race", "Circuit").distinct().toPandas()
rr = spark.table(RESULTS).toPandas()
rr["Day"] = pd.to_datetime(rr["EventDate"]).dt.date

# Ergast circuit name -> the Circuit string already used in the master table
circ_map = (rr.merge(master_keys, on=["Year", "Race"]).drop_duplicates("CircuitName").set_index("CircuitName")["Circuit"].to_dict())
done = set(zip(master_keys["Year"], master_keys["Race"]))
print(f"{len(done)} races already in the master table; mapped {len(circ_map)} circuits")

In [ ]:
def build_race(s, res):
    key = s.session_key
    laps = get("laps", session_key=key)
    if laps.empty:
        return None
    drv = get("drivers", session_key=key).drop_duplicates("driver_number").set_index("driver_number")
    st = get("stints", session_key=key)
    pits = get("pit", session_key=key)
    rc = get("race_control", session_key=key)
    wx = get("weather", session_key=key)

    t0 = pd.Timestamp(s.date_start)
    laps["Start"] = pd.to_datetime(laps["date_start"], format="ISO8601", utc=True)
    laps = laps.sort_values(["driver_number", "lap_number"]).reset_index(drop=True)
    end = laps.groupby("driver_number")["Start"].shift(-1).fillna(laps["Start"] + pd.to_timedelta(laps["lap_duration"], unit="s"))
    # a lap with no end (car retired on it) is kept with empty Time/Position, as FastF1 does, so the car shows as OUT
    laps["End"] = end
    laps["Time"] = (laps["End"] - t0).dt.total_seconds()
    laps["Position"] = laps.groupby("lap_number")["Time"].rank(method="first")
    laps["Driver"] = laps["driver_number"].map(drv["name_acronym"])

    for c in ("Stint", "TyreLife"):
        laps[c] = np.nan
    laps["Compound"], laps["FreshTyre"] = None, None
    for r in st.itertuples():
        sel = (laps["driver_number"] == r.driver_number) & laps["lap_number"].between(r.lap_start, r.lap_end)
        age0 = 0 if pd.isna(r.tyre_age_at_start) else r.tyre_age_at_start
        laps.loc[sel, "Stint"] = r.stint_number
        laps.loc[sel, "Compound"] = r.compound
        laps.loc[sel, "TyreLife"] = age0 + laps.loc[sel, "lap_number"] - r.lap_start + 1
        laps.loc[sel, "FreshTyre"] = bool(age0 == 0)

    out_lap = laps["is_pit_out_lap"].fillna(False).astype(bool) & (laps["lap_number"] > 1)
    pit_dur = {} if pits.empty else {(r.driver_number, int(r.lap_number) + 1): r.pit_duration for r in pits.itertuples() if pd.notna(r.lap_number)}
    sc = sc_laps(rc, int(laps["lap_number"].max()))

    df = pd.DataFrame({
        "Time": laps["Time"], "Driver": laps["Driver"], "DriverNumber": laps["driver_number"].astype(int).astype(str),
        "LapTime": laps["lap_duration"], "LapNumber": laps["lap_number"].astype(float), "Stint": laps["Stint"],
        "Sector1Time": laps["duration_sector_1"], "Sector2Time": laps["duration_sector_2"], "Sector3Time": laps["duration_sector_3"],
        "SpeedI1": laps["i1_speed"], "SpeedI2": laps["i2_speed"], "SpeedST": laps["st_speed"],
        "Compound": laps["Compound"], "TyreLife": laps["TyreLife"], "FreshTyre": laps["FreshTyre"],
        "Team": laps["driver_number"].map(drv["team_name"]), "LapStartTime": (laps["Start"] - t0).dt.total_seconds(),
        "LapStartDate": laps["Start"], "TrackStatus": np.where(sc[laps["lap_number"].astype(int)], "4", "1"),
        "Position": laps["Position"], "End": laps["End"].fillna(laps["Start"]).fillna(t0),
    })
    df["PitOutTime"] = np.where(out_lap, df["LapStartTime"], np.nan)
    df["PitInTime"] = np.where(out_lap.groupby(laps["driver_number"]).shift(-1).fillna(False).astype(bool), df["Time"], np.nan)
    df["HasPit"] = out_lap
    df["PitDuration"] = [pit_dur.get((n, int(l)), 0.0) if o else 0.0 for n, l, o in zip(laps["driver_number"], laps["lap_number"], out_lap)]
    df["PitDuration"] = pd.to_numeric(df["PitDuration"], errors="coerce").fillna(0.0)

    if not wx.empty:
        wx["End"] = pd.to_datetime(wx["date"], format="ISO8601", utc=True)
        wx = wx.sort_values("End").rename(columns={"air_temperature": "AirTemp_C", "humidity": "Humidity_pct", "pressure": "Pressure", "rainfall": "Rainfall_mm",
                                                   "track_temperature": "TrackTemp_C", "wind_direction": "WindDirection", "wind_speed": "WindSpeed_kmh"})
        df = pd.merge_asof(df.sort_values("End"), wx[["End", "AirTemp_C", "Humidity_pct", "Pressure", "Rainfall_mm", "TrackTemp_C", "WindDirection", "WindSpeed_kmh"]], on="End", direction="nearest")
        df["Rainfall_mm"] = df["Rainfall_mm"].astype(float) > 0

    by_pos = df.sort_values(["LapNumber", "Position"])
    df["GapToAhead"] = by_pos.groupby("LapNumber")["Time"].diff().reindex(df.index)
    df["DeltaToLeader"] = df["Time"] - df.groupby("LapNumber")["Time"].transform("min")
    df["DeltaToAverage"] = df["LapTime"] - df.groupby("LapNumber")["LapTime"].transform("mean")
    df["RaceLap"] = df["LapNumber"]

    df["Year"], df["Race"] = int(res["Year"].iloc[0]), res["Race"].iloc[0]
    cname = res["CircuitName"].iloc[0]
    df["Circuit"] = circ_map.get(cname, cname)
    if cname not in circ_map:
        print(f"   new circuit (no earlier race to match): {cname}")
    df["Location"], df["Country"] = s.location, s.country_name
    df["SessionDate"] = pd.Timestamp(res["EventDate"].iloc[0])
    df["TeamName"] = df["Team"]
    df["TeamColor"] = df["DriverNumber"].astype(int).map(drv["team_colour"])
    df["BroadcastName"] = df["DriverNumber"].astype(int).map(drv["broadcast_name"])
    r = res.set_index("Driver")
    for col, src in (("QualiPosition", "QualiPosition"), ("FinalPosition", "FinalPosition"), ("Status", "Status"), ("Points", "Points")):
        df[col] = df["Driver"].map(r[src])

    types = {f.name: f.dataType.simpleString() for f in schema.fields}
    rows = [tuple(cast(v, types[c]) for c, v in zip(types, vals)) for vals in df.reindex(columns=list(types)).itertuples(index=False, name=None)]
    return spark.createDataFrame(rows, schema)

In [ ]:
added = 0
for year in range(FIRST_YEAR, pd.Timestamp.now().year + 1):
    ses = get("sessions", year=year, session_name="Race")
    if ses.empty:
        continue
    ses = ses[~ses.get("is_cancelled", pd.Series(False, index=ses.index)).fillna(False).astype(bool)]
    for s in ses.itertuples():
        if pd.Timestamp(s.date_start) > pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=4):
            continue
        day = pd.Timestamp(s.date_start).date()
        res = rr[(rr["Year"] == year) & rr["Day"].apply(lambda d: abs((d - day).days) <= 1)]
        if res.empty:
            print(f"skip {year} {s.location}: no results yet in {RESULTS} (run phase_1b_results first)")
            continue
        race = res["Race"].iloc[0]
        if (year, race) in done:
            continue
        sdf = build_race(s, res)
        if sdf is None:
            print(f"skip {year} {race}: OpenF1 has no laps yet")
            continue
        sdf.write.mode("append").saveAsTable(MASTER)
        done.add((year, race))
        added += 1
        print(f"added {year} {race}: {sdf.count()} laps")
print("New races added:", added)